# 19 · Recursion and Function Design

**Level 2 · Data Structures and Functions** &nbsp;|&nbsp; Previous: [18_Function_Arguments_and_Scope](18_Function_Arguments_and_Scope.ipynb) &nbsp;|&nbsp; Next: Level 3, `20_Errors_and_Debugging` *(coming soon)*

This notebook ends Level 2. It has two parts: **recursion** (functions that call themselves, *optional*) and **function design** (how to write small, clear functions and how to break a longer program into them). Part B does not need Part A, so if recursion feels like too much right now, jump straight to section 7.

### 🎯 Learning goals

By the end of this notebook you will be able to:

- explain recursion in terms of a **base case** and a **recursive case**, and write simple recursive functions *(optional)*
- follow the **call stack** to see what a recursive function does step by step *(optional)*
- decide when recursion is a good idea and when a loop is better *(optional)*
- write **small, single-purpose functions** with good names that return values instead of printing them
- **decompose** a problem into helper functions, and turn a long script into a set of functions

In [ ]:
import sys

sys.path.insert(0, "../src")
from course_tools import check, check_output

## 1 · Why this matters

You now know how to write a function (notebook 17) and how to pass information in and out of it (notebook 18). Two questions remain.

First: *what if a problem looks like a smaller copy of itself?* Counting down from 5 is "say 5, then count down from 4". Adding up the digits of 472 is "take the 2, then add up the digits of 47". A function that solves the problem by calling itself on the smaller copy is **recursive**. It is a way of thinking, not a necessity: everything recursion does, a loop can do too. But for some problems it is the clearest way to say what you mean.

Second, and far more important in daily work: *how do you write functions that are pleasant to use, test and change?* A 60-line program that does everything in one block works, until you need to change one part of it. Programs made of small functions, each with one job and a good name, are easier to read, fix and reuse. That skill matters in every program you will ever write.

**Part A (sections 2 to 6, optional): Recursion.** **Part B (sections 7 to 9): Function design.** Then common mistakes, exercises and the Level 2 recap.

# Part A · Recursion *(optional)*

## 2 · A function that calls itself

Imagine a set of Russian nesting dolls. To find the smallest one you open the doll, and inside is a smaller set of dolls. You repeat the same action on the smaller set, until you open a doll with nothing inside.

A **recursive function** is a function that calls itself with a *smaller* version of the problem. Every recursive function needs two parts:

- the **base case**: the smallest problem, which you can answer directly without calling yourself again (the doll with nothing inside)
- the **recursive case**: everything else, where you do a little work and call yourself on a smaller problem

Here is a countdown:

In [ ]:
def countdown(n):
    if n == 0:                 # base case: nothing left to count
        print("Liftoff!")
    else:                      # recursive case: do a little work, then a smaller problem
        print(n)
        countdown(n - 1)


countdown(3)

`countdown(3)` prints `3` and then asks `countdown(2)` to do the rest. That prints `2` and asks `countdown(1)`, and so on, until `countdown(0)` prints `Liftoff!` and nothing more is called.

Two rules keep recursion safe:

1. There must be a **base case** that stops the chain.
2. Every recursive call must move **closer to the base case** (here `n - 1` gets closer to `0`).

### Try it

Write `count_up(n)` that prints `1, 2, ... n` on separate lines **recursively** (no loop). Hint: call `count_up(n - 1)` *first*, then print `n`. What happens to the order if you print first instead?

In [ ]:
# your code here

## 3 · Factorial and the call stack

The **factorial** of `n`, written `n!`, is `n × (n-1) × ... × 1`. So `5! = 5 × 4 × 3 × 2 × 1 = 120`. Notice that `5! = 5 × 4!`. The big problem contains a smaller copy of itself, which is exactly the shape recursion likes.

In [ ]:
def factorial(n):
    if n <= 1:                         # base case: 1! = 1 (and 0! = 1)
        return 1
    return n * factorial(n - 1)        # recursive case


print(factorial(5))
print(factorial(10))

How does Python keep track of all these unfinished calls? Each call gets its own **local variables** (notebook 18): the `n` in `factorial(5)` is a different variable from the `n` in `factorial(4)`. Python stacks the calls on top of each other like a pile of plates. This pile is the **call stack**. A call can only finish once the call above it has returned its value.

The function below prints what happens. The indentation shows how deep in the stack we are:

In [ ]:
def factorial_traced(n, depth=0):
    indent = "    " * depth
    print(f"{indent}factorial({n}) called")
    if n <= 1:
        print(f"{indent}base case, returns 1")
        return 1
    result = n * factorial_traced(n - 1, depth + 1)
    print(f"{indent}factorial({n}) returns {result}")
    return result


factorial_traced(4)

Read it in two halves. On the way **down**, each call waits for the next one (4, 3, 2, 1). At the bottom, the base case returns `1`. On the way **back up**, each waiting call finishes: `2 × 1 = 2`, then `3 × 2 = 6`, then `4 × 6 = 24`.

The same job with a loop, for comparison:

In [ ]:
def factorial_loop(n):
    result = 1
    for k in range(2, n + 1):
        result *= k
    return result


print(factorial_loop(5))

Both are correct. The recursive version reads like the definition (`n! = n × (n-1)!`); the loop version is a little more mechanical.

### Try it

Write a recursive `total_to(n)` that returns `1 + 2 + ... + n`. What is the base case? What does `total_to(n)` equal in terms of `total_to(n - 1)`?

In [ ]:
# your code here

## 4 · Fibonacci: recursion that branches

The Fibonacci numbers start `0, 1, 1, 2, 3, 5, 8, 13, ...`. Each one is the sum of the two before it. This time there are **two** base cases and **two** recursive calls:

In [ ]:
def fib(n):
    if n < 2:                          # base cases: fib(0) = 0, fib(1) = 1
        return n
    return fib(n - 1) + fib(n - 2)


for i in range(10):
    print(fib(i), end=" ")
print()

It works, but there is a catch. `fib(5)` calls `fib(4)` and `fib(3)`; `fib(4)` calls `fib(3)` again and `fib(2)`; and so on. The same values get computed over and over. Let's count the calls. (We use `global` here just so the function can change a counter defined outside it.)

In [ ]:
calls = 0


def fib_counted(n):
    global calls
    calls += 1
    if n < 2:
        return n
    return fib_counted(n - 1) + fib_counted(n - 2)


for n in [5, 10, 15, 20, 25]:
    calls = 0
    answer = fib_counted(n)
    print(f"fib({n}) = {answer}, needed {calls} calls")

The number of calls explodes: asking for `fib(35)` would need about 30 million calls, and `fib(50)` would take hours. A loop does the same job with just `n` steps:

In [ ]:
def fib_loop(n):
    previous, current = 0, 1
    for _ in range(n):
        previous, current = current, previous + current
    return previous


print(fib_loop(10), fib_loop(50))

So for Fibonacci the plain recursive version is a good *illustration* of recursion but a poor *tool*. *(Preview: Level 6 shows how to keep the recursive shape and make it fast by remembering earlier answers. We won't need that here.)*

### Try it

Using `fib` and a `for` loop, print the first 12 Fibonacci numbers. Then call `fib_counted(12)` and read `calls` to see how many calls it needed.

In [ ]:
# your code here

## 5 · Recursion on data

Recursion works on data too: "a list is its first item plus a smaller list". Slicing gives us the smaller list: `numbers[1:]` is everything except the first item.

In [ ]:
def sum_list(numbers):
    if len(numbers) == 0:              # base case: an empty list adds up to 0
        return 0
    return numbers[0] + sum_list(numbers[1:])


print(sum_list([4, 8, 15, 16]))
print(sum_list([]))

Whenever you design a recursive function, ask three questions:

1. **Base case:** what is the smallest input, and what is its answer? (`[]` gives `0`.)
2. **Shrink:** how do I make the input smaller? (drop the first item)
3. **Combine:** how do I build the answer from the smaller answer? (add the first item to it)

(Of course the built-in `sum()` does this job, and you would use it. The point is to see the shape.)

### Try it

Write a recursive `count_letter(text, letter)` that returns how many times `letter` appears in `text`. Base case: the empty string. Shrink with `text[1:]`.

In [ ]:
# your code here

## 6 · When recursion is (and isn't) a good idea

**Recursion shines when:**

- the data is **nested** or tree-shaped (a list containing lists containing lists; folders inside folders), because the structure itself repeats
- the problem naturally splits into smaller copies of itself (you will meet this in sorting and searching algorithms later in the course)
- the recursive version is much clearer than any loop you can think of

**A loop is usually better when:**

- the job is simply "repeat N times" or "go through each item" (`factorial`, `sum_list` and `countdown` above are really loops in disguise)
- the recursion would recompute the same things again and again (Fibonacci)
- the input can be large, because of the next point.

### Python limits how deep recursion can go

Every call uses a bit of memory on the call stack, so Python stops you at a **recursion limit**:

In [ ]:
print(sys.getrecursionlimit())

(`sys.setrecursionlimit()` can change it, but raising it is rarely the right fix: it is usually a sign that a loop would be better.) Recursion deeper than the limit fails with a `RecursionError`, which is Python's way of saying "this is going too deep, probably an infinite recursion": ⚠️ **Expected to fail.**

In [ ]:
print(len(str(factorial(500))), "digits")    # fine
print(len(str(factorial(1500))), "digits")   # too deep for the call stack

The loop version has no such limit:

In [ ]:
print(len(str(factorial_loop(1500))), "digits")

**Rule of thumb for beginners:** write a loop unless the problem is clearly recursive in shape (nested data, "a smaller copy of the same problem"). When you do use recursion, always check the base case first.

That is the end of the optional recursion material. Everything from here on works fine without it.

# Part B · Function design

## 7 · Small, single-purpose functions

A good function does **one job**, and its name says what the job is. A handy test: describe what the function does in one sentence. If the sentence needs the word "and", you probably have two functions.

Here is a function that does too much: it computes an average *and* prints it:

In [ ]:
def show_average(numbers):
    total = 0
    for n in numbers:
        total += n
    print("Average:", total / len(numbers))


show_average([4, 8, 15, 16])

It works, but what if you want the average to compare two groups, or round it, or put it in a sentence? The value was printed and then lost. Compare:

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)


def show_average(numbers):
    print("Average:", average(numbers))


morning = [4, 8, 15, 16]
evening = [10, 20, 30]
show_average(morning)
print("Evening is higher:", average(evening) > average(morning))

Now `average` has one job (calculate), `show_average` has one job (display), and `average` can be reused anywhere. A useful habit: **separate calculating from showing**. Functions that *return* values can be combined, compared and tested. Functions that only *print* cannot (remember `return` vs `print` from notebook 17).

More habits of good functions:

- **A verb-based name** that says what it does: `average`, `is_palindrome`, `count_vowels`. Yes/no functions often start with `is_` or `has_`.
- **A docstring** (notebook 17) that says what goes in and what comes out.
- **Few parameters.** If you need six, something is probably wrong with the design.
- **Take what you need as arguments, return what you produce.** Avoid reading or changing global variables inside the function (notebook 18).
- **Short.** If you can't see the whole function on one screen, look for a piece to move out.

In [ ]:
def is_vowel(letter):
    """Return True if letter is a vowel (a, e, i, o, u), in any case."""
    return letter.lower() in "aeiou"


def count_vowels(text):
    """Return how many vowels text contains."""
    count = 0
    for ch in text:
        if is_vowel(ch):
            count += 1
    return count


print(count_vowels("Function Design"))

Notice that `count_vowels` is easy to read because `is_vowel` has a name. The helper also works on its own: `is_vowel("E")`.

### Try it

Write `is_even(n)` (returns `True`/`False`) and `count_evens(numbers)` that uses it. Test each one separately by calling it with a few inputs.

In [ ]:
# your code here

## 8 · Decomposing a problem

**Decomposing** means splitting a big problem into small ones until each piece is easy. The usual method is **top-down**:

1. Say what the whole program must produce.
2. List the steps in plain words.
3. Turn each step into a function. If a step is still hard, split it again.

Problem: *given a sentence, report how many words it has, the longest word, and the average word length.*

Steps: (1) split the sentence into words, without punctuation; (2) find the longest word; (3) find the average length; (4) put the answers into a message. Each step is a small function:

In [ ]:
def get_words(sentence):
    """Return the words of sentence with surrounding punctuation removed."""
    words = []
    for raw in sentence.split():
        words.append(raw.strip(".,!?;:"))
    return words


def longest_word(words):
    """Return the longest word (the first one if there is a tie)."""
    best = words[0]
    for word in words:
        if len(word) > len(best):
            best = word
    return best


def average_length(words):
    """Return the average number of letters per word."""
    total = 0
    for word in words:
        total += len(word)
    return total / len(words)


def describe(sentence):
    """Return a one-line summary of sentence."""
    words = get_words(sentence)
    return (
        f"{len(words)} words, longest: {longest_word(words)}, "
        f"average length {average_length(words):.1f}"
    )


print(describe("The quick brown fox jumps over the lazy dog."))

A big advantage: you can **test each piece in isolation**. If the output is wrong, you can call the helpers one at a time and see which one is at fault:

In [ ]:
print(get_words("Hello, world!"))
print(longest_word(["a", "abc", "ab"]))
print(average_length(["ab", "abcd"]))

`describe` is the "boss" function: it doesn't do the details, it just calls the helpers in the right order. Reading `describe` tells you *what* the program does; reading the helpers tells you *how*.

### Try it

Extend the example: write `count_long_words(words, size)` that returns how many words have at least `size` letters, and add it to the summary in `describe`.

In [ ]:
# your code here

## 9 · Breaking a longer program into functions

Often you start by writing everything in one block, which is fine for a first draft. Here is such a script. It works, but notice the repeated loops and the "parts" it has:

In [ ]:
readings = [18.5, 22.0, 31.2, 28.4, 35.1, 19.9]

# convert to Fahrenheit
fahrenheit = []
for c in readings:
    fahrenheit.append(c * 9 / 5 + 32)

# count hot days
hot = 0
for c in readings:
    if c > 30:
        hot += 1

# print the summary
print("Readings:", len(readings))
print(f"Hottest: {max(readings)} C = {max(fahrenheit):.1f} F")
print("Days above 30 C:", hot)

**Signs that code wants to be split into functions:**

- comments that act as section titles (`# convert to Fahrenheit`): each such section is a function waiting to be named
- the same kind of loop written more than once
- you can't describe the script in one sentence without "and"
- you want to run one part again with different data

The method: take each commented section, give it a name, decide what goes **in** (parameters) and what comes **out** (return value), and finish with a short `main()` that calls them in order. Calculating functions return; only the display function prints.

In [ ]:
def to_fahrenheit(celsius):
    """Convert one temperature from Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


def count_above(values, limit):
    """Return how many items in values are greater than limit."""
    count = 0
    for value in values:
        if value > limit:
            count += 1
    return count


def print_summary(readings):
    """Print a short summary of a list of Celsius readings."""
    hottest = max(readings)
    print("Readings:", len(readings))
    print(f"Hottest: {hottest} C = {to_fahrenheit(hottest):.1f} F")
    print("Days above 30 C:", count_above(readings, 30))


def main():
    readings = [18.5, 22.0, 31.2, 28.4, 35.1, 19.9]
    print_summary(readings)


main()

Same output, but now: `to_fahrenheit` and `count_above` can be reused for any data (`count_above(scores, 50)`), each can be tested alone, and `main` reads like a table of contents. (Notice we also dropped the whole `fahrenheit` list, because we only needed the hottest value converted. Refactoring often makes code simpler too.)

The usual layout of a script: **imports, then function definitions, then a call to `main()` at the bottom.** Python needs a function to be defined before the line that *calls* it runs, but a function body can refer to functions defined later in the file.

### Try it

Add a function `coldest_day_message(readings)` that returns a string like `"Coldest: 18.5 C"` and call it from `print_summary`.

In [ ]:
# your code here

## 10 · Common mistakes

### 1. Recursion without a base case

Nothing stops the calls, so Python gives up at the recursion limit: ⚠️ **Expected to fail.**

In [ ]:
def countdown_broken(n):
    print(n)
    countdown_broken(n - 1)      # never stops: there is no base case


countdown_broken(3)

(The output starts `3, 2, 1, 0, -1, ...` and ends in a `RecursionError`.)

### 2. A recursive call that doesn't get closer to the base case

The base case exists, but the argument moves the wrong way (or not at all): ⚠️ **Expected to fail.**

In [ ]:
def total_to_broken(n):
    if n == 0:
        return 0
    return n + total_to_broken(n + 1)     # oops: n + 1 moves away from 0


total_to_broken(3)

### 3. Forgetting to `return` the recursive result

Without `return`, the function does the work and then hands back `None`. The next call up the stack then tries to multiply by `None`: ⚠️ **Expected to fail.**

In [ ]:
def factorial_no_return(n):
    if n <= 1:
        return 1
    n * factorial_no_return(n - 1)       # computed, then thrown away


print(factorial_no_return(5))

### 4. Printing where you should return

If a function prints its answer, the caller gets `None` back and can't use it: ⚠️ **Expected to fail.**

In [ ]:
def show_square(n):
    print(n * n)


result = show_square(4)
print(result)
print(result + 1)

### 5. A function that does too many things

`process_everything(data)` that reads, cleans, calculates and prints can only be used as one block. If you find a function hard to name, or its name contains "and", split it.

### 6. Surprising changes to an argument (aliases again)

A list passed to a function is the *same* list (an alias, notebook 12). If your helper changes it, the caller's data changes too:

In [ ]:
def top_two(scores):
    scores.sort(reverse=True)       # sorts the caller's list in place!
    return scores[:2]


results = [70, 95, 82, 60]
print(top_two(results))
print(results)                      # the original order is gone

A function should not change its arguments unless that is its clear purpose (and its name and docstring say so). `sorted(scores, reverse=True)` returns a *new* sorted list and leaves the original alone.

### 7. Using recursion where a loop is simpler

If you write a recursive function and the recursive call is just "do the same thing for the next number", a `for` loop is almost always shorter, faster and has no depth limit.

## 11 · Exercises

Exercises 19.1 to 19.5 are about **function design** and use no recursion. Exercises 19.6 to 19.11 are the **optional recursion** exercises.

### Exercise 19.1 · Warm-up: a tiny helper

Write `is_vowel(letter)` that returns `True` if `letter` is a vowel (`a`, `e`, `i`, `o`, `u`, in either case) and `False` otherwise.

In [ ]:
def is_vowel(letter):
    # your code here
    pass

In [ ]:
check(is_vowel("a"), True)
check(is_vowel("E"), True)
check(is_vowel("z"), False)
check(is_vowel("Y"), False)

### Exercise 19.2 · Warm-up: sum of digits

Write `sum_of_digits(n)` that returns the sum of the digits of a non-negative whole number, **using a loop** (no recursion). For example `sum_of_digits(472)` is `4 + 7 + 2 = 13`.

<details><summary>Hint</summary>

`n % 10` is the last digit and `n // 10` is the number without its last digit. Repeat while `n > 0`.

</details>

In [ ]:
def sum_of_digits(n):
    # your code here
    pass

In [ ]:
check(sum_of_digits(472), 13)
check(sum_of_digits(0), 0)
check(sum_of_digits(7), 7)
check(sum_of_digits(505), 10)
check(sum_of_digits(99999), 45)

### Exercise 19.3 · Practice: palindrome checker

A **palindrome** reads the same forwards and backwards, ignoring case, spaces and punctuation: `"A man, a plan, a canal: Panama"`.

Build it from two small functions:

- `clean(text)` returns the text in lowercase with only letters and digits kept (use `.isalnum()`), e.g. `clean("A man, a plan!")` is `"amanaplan"`
- `is_palindrome(text)` returns `True` or `False`, and uses `clean`

<details><summary>Hint</summary>

`word[::-1]` is `word` reversed. Compare the cleaned text with its reverse.

</details>

In [ ]:
def clean(text):
    # your code here
    pass


def is_palindrome(text):
    # your code here
    pass

In [ ]:
check(clean("A man, a plan!"), "amanaplan")
check(is_palindrome("A man, a plan, a canal: Panama"), True)
check(is_palindrome("racecar"), True)
check(is_palindrome("Hello"), False)
check(is_palindrome(""), True)

### Exercise 19.4 · Practice: break a longer program into functions

Here is a script that reports on one student's test scores. It works, but it is one long block:

```python
name = "Ana"
scores = [92, 85, 78]

total = 0
for s in scores:
    total += s
avg = total / len(scores)

if avg >= 90:
    grade = "A"
elif avg >= 80:
    grade = "B"
elif avg >= 70:
    grade = "C"
elif avg >= 60:
    grade = "D"
else:
    grade = "F"

print(f"{name}: average {avg:.1f}, grade {grade}, best {max(scores)}")
```

Split it into three functions, each with one job:

- `average(scores)` returns the average (a float)
- `letter_grade(avg)` returns `"A"`, `"B"`, `"C"`, `"D"` or `"F"` using the same cut-offs as above
- `report(name, scores)` **returns** the summary line (e.g. `"Ana: average 85.0, grade B, best 92"`) and uses the other two. It does not print.

In [ ]:
def average(scores):
    # your code here
    pass


def letter_grade(avg):
    # your code here
    pass


def report(name, scores):
    # your code here
    pass

In [ ]:
check(average([80, 90, 100]), 90.0)
check(letter_grade(85.0), "B")
check(letter_grade(59.9), "F")
check(letter_grade(90.0), "A")
check(report("Ana", [92, 85, 78]), "Ana: average 85.0, grade B, best 92")
check(report("Ben", [50, 60, 70]), "Ben: average 60.0, grade D, best 70")

### Exercise 19.5 · Stretch: an interactive program in functions

Write a program, organised into functions, that works like this (the text in quotes is what the user types):

```
How many numbers? 3
Number: abc            <- not a whole number
Please enter a whole number.
Number: 4
Number: 10
Number: 6
Total: 20
Largest: 10
```

- `read_whole_number(prompt)` asks with `input(prompt)` until the text is made of digits (`.isdigit()`), printing `Please enter a whole number.` after each bad answer, and returns the answer as an `int`
- `main()` asks `"How many numbers? "`, reads that many numbers with `read_whole_number("Number: ")`, then prints `Total: ...` and `Largest: ...`. If the count is `0`, it prints `No numbers.` instead.
- the last line of the cell calls `main()`

Only `read_whole_number` talks to the user about bad input; `main` just uses it.

In [ ]:
def read_whole_number(prompt):
    # your code here
    pass


def main():
    # your code here
    pass


main()

In [ ]:
check_output(
    "Please enter a whole number.\nPlease enter a whole number.\nPlease enter a whole number.\nTotal: 20\nLargest: 10",
    inputs=["3", "abc", "4", "10", "x", "y", "6"],
)
check_output("No numbers.", inputs=["0"])

### Optional: recursion exercises

The next six exercises are about recursion. Skip them if you skipped Part A.

### Exercise 19.6 · Warm-up: factorial (optional, recursion)

Write `factorial(n)` **recursively**. `factorial(0)` and `factorial(1)` are `1`. `factorial(5)` is `120`.

In [ ]:
def factorial(n):
    # your code here
    pass

In [ ]:
check(factorial(0), 1)
check(factorial(1), 1)
check(factorial(5), 120)
check(factorial(10), 3628800)

### Exercise 19.7 · Practice: power (optional, recursion)

Write `power(base, exponent)` **recursively**, without using `**`. Assume `exponent` is a whole number that is `0` or more. Think: `base` to the power `exponent` is `base` times `base` to the power `exponent - 1`. What is the base case?

In [ ]:
def power(base, exponent):
    # your code here
    pass

In [ ]:
check(power(2, 10), 1024)
check(power(5, 0), 1)
check(power(3, 1), 3)
check(power(7, 2), 49)

### Exercise 19.8 · Practice: sum of digits, recursively (optional, recursion)

Write `sum_digits(n)` **recursively** (no loop), so `sum_digits(472)` is `13`. Use `n % 10` (last digit) and `n // 10` (the rest). The base case is a number with a single digit.

In [ ]:
def sum_digits(n):
    # your code here
    pass

In [ ]:
check(sum_digits(472), 13)
check(sum_digits(0), 0)
check(sum_digits(7), 7)
check(sum_digits(99999), 45)

### Exercise 19.9 · Practice: reverse a string (optional, recursion)

Write `reverse(text)` **recursively**, so `reverse("abc")` is `"cba"`. Do not use slicing with `[::-1]`. Idea: the reverse of a word is the reverse of everything after the first letter, followed by the first letter. The base case is the empty string.

In [ ]:
def reverse(text):
    # your code here
    pass

In [ ]:
check(reverse("abc"), "cba")
check(reverse(""), "")
check(reverse("a"), "a")
check(reverse("Python"), "nohtyP")

### Exercise 19.10 · Stretch: Fibonacci two ways (optional, recursion)

Write both `fib(n)` (recursive) and `fib_loop(n)` (a loop). Both return the n-th Fibonacci number, with `fib(0) = 0` and `fib(1) = 1`. Then try `fib(30)` and `fib_loop(30)` in a scratch cell and notice which one is slow.

In [ ]:
def fib(n):
    # your code here
    pass


def fib_loop(n):
    # your code here
    pass

In [ ]:
check(fib(0), 0)
check(fib(1), 1)
check(fib(10), 55)
check(fib(15), 610)
check(fib_loop(0), 0)
check(fib_loop(10), 55)
check(fib_loop(50), 12586269025)

### Exercise 19.11 · Stretch: flatten nested lists (optional, recursion)

Write `flatten(items)` that takes a list which may contain other lists (nested to any depth) and returns one flat list of the non-list items, in order. For example `flatten([1, [2, 3], [4, [5, 6]]])` is `[1, 2, 3, 4, 5, 6]`.

This is a problem where recursion is much easier than a loop: whenever an item is itself a list, `flatten` calls itself on it.

<details><summary>Hint</summary>

Check whether an item is a list with `type(item) is list`. Build a `result` list; for a list item, `extend` the result with `flatten(item)`; otherwise `append` the item.

</details>

In [ ]:
def flatten(items):
    # your code here
    pass

In [ ]:
check(flatten([1, [2, 3], [4, [5, 6]]]), [1, 2, 3, 4, 5, 6])
check(flatten([]), [])
check(flatten([[[]]]), [])
check(flatten([1, 2]), [1, 2])
check(flatten([[1], [[2]], 3]), [1, 2, 3])

## 12 · Recap of Level 2

**This notebook**

- **Recursion** *(optional)*: a function that calls itself on a smaller problem. It needs a **base case** (stop) and a **recursive case** (shrink and combine). Calls pile up on the **call stack** and finish in reverse order. Python limits the depth (`RecursionError`).
- Use recursion for nested or tree-shaped data and problems that split into smaller copies; use loops for simple repetition and when the same work would be repeated (Fibonacci).
- **Good functions** have one job and a verb-based name, take arguments and return values, and calculate separately from printing.
- **Decompose** top-down: say what the whole must do, list the steps, make each step a function. Break up scripts whose sections are separated by comments, then finish with a short `main()`.
- Don't change arguments unless that is the function's declared purpose.

**Level 2 as a whole**

- **Lists** (12) store ordered, mutable items; **tuples** (13) are immutable and support unpacking; **dictionaries** (14) map keys to values; **sets** (15) hold unique items. **Choosing the right collection** (16) is a question of order, uniqueness, lookup by key, and whether the data should change.
- **Functions** (17) package code for reuse with `def`, `return` and docstrings; **arguments and scope** (18) cover positional, keyword and default arguments, `*args`/`**kwargs`, local vs. global names, and the mutable-default pitfall.
- With this notebook you can organise both your **data** (collections) and your **code** (small functions).

### What's next: Level 3, Practical Python

Level 3 turns your programs into robust, practical tools: finding and fixing bugs, handling errors gracefully, testing, working with files and modules. It begins with understanding what went wrong when code fails.

**Next up:** `20_Errors_and_Debugging` *(coming soon)*, the first notebook of Level 3.